[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/duckdb-certified/notebooks/day-08-performance-profiling.ipynb#scrollTo=11223344)

---
# Day 8 · Performance and Query Profiling — EXPLAIN ANALYZE and Parallel Execution
**certified-journeys / duckdb-certified** &nbsp;|&nbsp; Performance & Profiling

> **Goal for today:** Use EXPLAIN and EXPLAIN ANALYZE to inspect query plans, identify slow operators, and measure the impact of optimisations like predicate pushdown and column pruning.

---
## Why profiling matters

DuckDB is fast by default, but writing *analytically efficient* SQL still matters. The profiling tools show you exactly what the query engine is doing:

| Tool | What it shows |
|---|---|
| `EXPLAIN` | Logical + physical plan: operators, join order, projections |
| `EXPLAIN ANALYZE` | Everything EXPLAIN shows + actual timing and row counts |
| `PRAGMA enable_profiling` | Write JSON profile to file after every query |
| `SET threads` | Control how many CPU threads DuckDB uses |
| `ANALYZE` | Refresh table statistics so the planner makes better decisions |

> **Read first:** [DuckDB query profiling docs](https://duckdb.org/docs/dev/profiling)

In [ ]:
%pip install -q duckdb

---
## Step 1 · Build a dataset large enough to profile

Small tables make every query instantaneous — we need enough rows that the planner's decisions actually affect runtime. We'll generate ~500 K rows across three tables.

In [ ]:
import duckdb, json, os

con = duckdb.connect()

# Customers — 10 000 rows
con.execute("""
CREATE TABLE customers AS
SELECT
    i                                              AS customer_id,
    'Customer_' || i                               AS name,
    CASE i % 5
        WHEN 0 THEN 'US'
        WHEN 1 THEN 'GB'
        WHEN 2 THEN 'DE'
        WHEN 3 THEN 'FR'
        ELSE 'CA'
    END AS country,
    (random() * 5 + 1)::INT AS tier  -- 1 to 5
FROM range(1, 10001) t(i)
""")

# Products — 1 000 rows
con.execute("""
CREATE TABLE products AS
SELECT
    i                                         AS product_id,
    'Product_' || i                           AS sku,
    CASE i % 8
        WHEN 0 THEN 'Electronics'
        WHEN 1 THEN 'Clothing'
        WHEN 2 THEN 'Books'
        WHEN 3 THEN 'Home'
        WHEN 4 THEN 'Sports'
        WHEN 5 THEN 'Food'
        WHEN 6 THEN 'Beauty'
        ELSE 'Toys'
    END AS category,
    round(random() * 500 + 5, 2) AS price
FROM range(1, 1001) t(i)
""")

# Orders — 500 000 rows
con.execute("""
CREATE TABLE orders AS
SELECT
    i                                              AS order_id,
    (random() * 9999 + 1)::INT                     AS customer_id,
    (random() * 999 + 1)::INT                      AS product_id,
    (random() * 9 + 1)::INT                        AS quantity,
    current_date - (random() * 730)::INT           AS order_date,
    CASE (i % 4)
        WHEN 0 THEN 'shipped'
        WHEN 1 THEN 'pending'
        WHEN 2 THEN 'delivered'
        ELSE 'cancelled'
    END AS status
FROM range(1, 500001) t(i)
""")

for tbl in ['customers', 'products', 'orders']:
    n = con.execute(f"SELECT count(*) FROM {tbl}").fetchone()[0]
    print(f"{tbl:<12}: {n:>8,} rows")

**What just happened?**
- Three tables with 10K + 1K + 500K rows — enough to see real planner decisions
- `range(1, N) t(i)` is DuckDB's table-valued function for generating row sequences
- No statistics exist yet — DuckDB will estimate row counts without `ANALYZE`

---
## Step 2 · EXPLAIN — read the logical and physical plan

`EXPLAIN` shows the query plan DuckDB *intends* to execute. No data is read.

Key operators you'll see:

| Operator | Meaning |
|---|---|
| `SEQ_SCAN` | Full table scan |
| `FILTER` | Row predicate applied after scan |
| `HASH_JOIN` | Build a hash table from the smaller side, probe with the larger |
| `PROJECTION` | SELECT column list |
| `HASH_GROUP_BY` | Aggregation using a hash table |
| `ORDER_BY` | Sort |
| `TOP_N` | Sort + limit combined — more efficient than full sort |

In [ ]:
# EXPLAIN without ANALYZE — shows the query plan, no data is read
plan = con.execute("""
EXPLAIN
SELECT
    c.country,
    p.category,
    sum(o.quantity * p.price) AS total_revenue,
    count(distinct o.customer_id) AS unique_customers
FROM orders o
JOIN customers c USING (customer_id)
JOIN products  p USING (product_id)
WHERE o.status = 'delivered'
GROUP BY c.country, p.category
ORDER BY total_revenue DESC
""").fetchall()

print("Physical Plan:")
print("=" * 60)
for row in plan:
    # Each row is (type, plan_text) or just (plan_text,)
    print(row[-1])

**What just happened?**
- `EXPLAIN` returned a text representation of the physical plan — no query actually ran
- Look for `FILTER` on `status = 'delivered'` — DuckDB applies it close to the scan (predicate pushdown)
- `HASH_JOIN` confirms DuckDB chose a hash join — good for large build sides
- **This is free** — always run EXPLAIN on expensive queries before executing them in production

---
## Step 3 · EXPLAIN ANALYZE — actual vs estimated row counts

`EXPLAIN ANALYZE` runs the query and annotates the plan with:
- **Estimated rows** — what the planner predicted
- **Actual rows** — what really flowed through each operator
- **Timing** per operator

Large discrepancies (estimated << actual) usually mean stale or missing statistics. The fix is `ANALYZE tablename`.

In [ ]:
import time

query = """
SELECT
    c.country,
    p.category,
    sum(o.quantity * p.price) AS total_revenue,
    count(distinct o.customer_id) AS unique_customers
FROM orders o
JOIN customers c USING (customer_id)
JOIN products  p USING (product_id)
WHERE o.status = 'delivered'
GROUP BY c.country, p.category
ORDER BY total_revenue DESC
"""

# Time the query execution
t0 = time.perf_counter()
analyze_plan = con.execute(f"EXPLAIN ANALYZE {query}").fetchall()
t1 = time.perf_counter()

print(f"Query ran in {t1-t0:.3f}s")
print("=" * 60)
for row in analyze_plan:
    print(row[-1])

**What just happened?**
- `EXPLAIN ANALYZE` ran the query fully and annotated each operator with actual counts and timings
- Look for operators where `Rows: estimated=X, actual=Y` diverge significantly
- **Large divergences** indicate the planner is guessing — running `ANALYZE` will fix this
- The timing shows which operators dominate — that's where to focus optimisation effort

---
## Step 4 · ANALYZE — refresh table statistics

DuckDB collects statistics lazily. After loading large datasets, run `ANALYZE` to give the planner accurate row counts and column distributions.

```sql
ANALYZE orders;     -- single table
ANALYZE;            -- all tables in the database
```

In [ ]:
import time

# Run ANALYZE on all three tables to update statistics
t_analyze_start = time.perf_counter()
con.execute("ANALYZE orders")
con.execute("ANALYZE customers")
con.execute("ANALYZE products")
t_analyze_end = time.perf_counter()
print(f"ANALYZE completed in {t_analyze_end - t_analyze_start:.3f}s")

# Re-run EXPLAIN ANALYZE — estimates should now be closer to actuals
t0 = time.perf_counter()
analyze_plan_after = con.execute(f"EXPLAIN ANALYZE {query}").fetchall()
t1 = time.perf_counter()
print(f"\nQuery after ANALYZE: {t1-t0:.3f}s")
print("=" * 60)
for row in analyze_plan_after:
    print(row[-1])

**What just happened?**
- `ANALYZE` scanned each table and updated per-column statistics (min, max, distinct count, null fraction)
- With accurate statistics the planner may reorder joins or choose a different join algorithm
- **Run ANALYZE after every large load** — especially after loading months of historical data
- On very large tables, `ANALYZE` takes a minute or two but pays for itself in planner accuracy

---
## Step 5 · JSON profiling output with PRAGMA enable_profiling

For programmatic analysis, DuckDB can write a JSON profile after every query:

```sql
PRAGMA enable_profiling = 'json';
PRAGMA profiling_output = '/tmp/profile.json';
-- ... run your query ...
PRAGMA disable_profiling;
```

The JSON file contains the full plan with timing at every node — perfect for building dashboards or CI performance gates.

In [ ]:
import json, os

profile_path = '/tmp/duckdb_profile.json'

# Enable JSON profiling and set the output file
con.execute("PRAGMA enable_profiling = 'json'")
con.execute(f"PRAGMA profiling_output = '{profile_path}'")

# Run a query — the profile is written to the file automatically
con.execute("""
SELECT c.country, count(*) AS orders
FROM orders o
JOIN customers c USING (customer_id)
WHERE o.status = 'shipped'
GROUP BY c.country
ORDER BY orders DESC
""")

# Turn profiling off so it doesn't affect subsequent queries
con.execute("PRAGMA disable_profiling")

# Parse and inspect the JSON profile
if os.path.exists(profile_path):
    with open(profile_path) as f:
        profile = json.load(f)

    # Print top-level keys and overall timing
    print("Top-level keys:", list(profile.keys()))
    print(f"\nTotal execution time: {profile.get('timing', 'N/A')} s")

    # Walk the operator tree and print operator names + timing
    def print_operators(node, indent=0):
        name    = node.get('name', '?')
        timing  = node.get('timing', 0)
        rows    = node.get('cardinality', '?')
        print(f"{'  ' * indent}{name}  rows={rows}  time={timing:.6f}s")
        for child in node.get('children', []):
            print_operators(child, indent + 1)

    print("\nOperator tree:")
    if 'children' in profile:
        print_operators(profile)
    else:
        print(json.dumps(profile, indent=2)[:2000])
else:
    print(f"Profile file not found at {profile_path} — check PRAGMA paths on this system.")

**What just happened?**
- `PRAGMA enable_profiling = 'json'` switches to machine-readable output
- The JSON tree mirrors the physical plan — each node has `timing` and `cardinality`
- `print_operators()` walks the tree recursively — a pattern you'd adapt for dashboards
- **Always `disable_profiling` when done** — it adds overhead to every subsequent query

---
## Step 6 · Controlling parallel execution with SET threads

DuckDB uses all available CPU threads by default. You can restrict this to:
- **Avoid competing with other processes** in a shared environment
- **Reproduce single-threaded timing** for benchmarks
- **Understand the parallelism benefit** by comparing 1 vs N threads

> **Reference:** [DuckDB parallel execution — threads](https://duckdb.org/docs/guides/performance/threads)

In [ ]:
import time

# How many threads does DuckDB detect?
default_threads = con.execute("SELECT current_setting('threads')").fetchone()[0]
print(f"Default threads: {default_threads}")

# Benchmark query: aggregation over 500 K rows
bench_query = """
SELECT
    status,
    date_trunc('month', order_date) AS month,
    sum(quantity) AS units,
    count(*) AS orders
FROM orders
GROUP BY status, month
ORDER BY month, status
"""

timings = {}

for n_threads in [1, 2, max(int(default_threads), 2)]:
    con.execute(f"SET threads = {n_threads}")
    # Warm-up run (not counted)
    con.execute(bench_query)
    # Timed run
    t0 = time.perf_counter()
    rows = con.execute(bench_query).fetchall()
    elapsed = time.perf_counter() - t0
    timings[n_threads] = elapsed
    print(f"threads={n_threads:2d}  time={elapsed:.4f}s  rows={len(rows)}")

# Restore default
con.execute(f"SET threads = {default_threads}")
print(f"\nRestored to {default_threads} threads")

**What just happened?**
- `SET threads = N` limits parallelism for the current connection
- The speedup going from 1→2 threads is usually close to 2× on CPU-bound aggregations
- Beyond 4 threads the benefit plateaus for most single-query workloads
- **In production containers**, set threads explicitly to match the vCPU allocation — avoids over-subscription

---
## Step 7 · Identifying and fixing a slow query — predicate pushdown and column pruning

Two of the most impactful optimisations:

| Technique | What it does | SQL pattern |
|---|---|---|
| Predicate pushdown | Apply WHERE filters as early as possible — scan fewer rows | Put filters directly in the subquery / CTE |
| Column pruning | Read only the columns you need — less I/O | `SELECT a, b` instead of `SELECT *` |
| Join reordering | Build the hash table from the smaller table | DuckDB often does this automatically post-ANALYZE |

We'll compare a naïve version vs an optimised version and measure the difference.

In [ ]:
import time

# --- Slow version: SELECT * in subquery, filter applied after join ---
slow_query = """
SELECT
    x.country,
    sum(x.quantity * x.price) AS revenue
FROM (
    SELECT *
    FROM orders o
    JOIN customers c USING (customer_id)
    JOIN products  p USING (product_id)
) x
WHERE x.status = 'delivered'
  AND x.country = 'US'
GROUP BY x.country
"""

# --- Faster version: filter early, select only needed columns ---
fast_query = """
SELECT
    c.country,
    sum(o.quantity * p.price) AS revenue
FROM (
    SELECT customer_id, product_id, quantity
    FROM orders
    WHERE status = 'delivered'   -- filter pushed down before the join
) o
JOIN (
    SELECT customer_id, country
    FROM customers
    WHERE country = 'US'          -- filter pushed down before the join
) c USING (customer_id)
JOIN (
    SELECT product_id, price
    FROM products
) p USING (product_id)
GROUP BY c.country
"""

# Warm-up
con.execute(slow_query); con.execute(fast_query)

# Time both
RUNS = 3
for label, q in [('slow (SELECT *)', slow_query), ('fast (pruned)', fast_query)]:
    times = []
    for _ in range(RUNS):
        t0 = time.perf_counter()
        result = con.execute(q).fetchall()
        times.append(time.perf_counter() - t0)
    avg = sum(times) / RUNS
    print(f"{label:<26}  avg={avg*1000:.1f}ms  result={result}")

**What just happened?**
- The slow version materialises all columns in a subquery, then filters — more memory, more work
- The fast version pushes filters into the scan and reads only the columns needed for the join and aggregation
- **DuckDB's planner often does this automatically** — but writing explicit filters helps on complex queries with many CTEs
- Use `EXPLAIN ANALYZE` on both versions to confirm the planner is applying pushdown

---
## Challenge — profile, identify, and fix a slow query

Use the tools from today to investigate a deliberately inefficient query.

In [ ]:
# Challenge:
# The query below computes the top-10 customers by total spend.
# It is intentionally written in a suboptimal way.
#
# 1. Run EXPLAIN ANALYZE on it and identify the most expensive operator.
# 2. Rewrite it to be faster (hint: avoid SELECT * and apply filters early).
# 3. Time both versions (at least 3 runs each) and report the speedup.

slow_top_customers = """
SELECT customer_name, total_spend
FROM (
    SELECT
        c.name AS customer_name,
        sum(p.price * o.quantity) AS total_spend,
        c.tier,
        c.country
    FROM (
        SELECT *
        FROM orders
        JOIN customers USING (customer_id)
        JOIN products  USING (product_id)
    ) AS full_table
    LEFT JOIN customers c ON full_table.customer_id = c.customer_id
    LEFT JOIN products  p ON full_table.product_id  = p.product_id
    LEFT JOIN orders    o ON full_table.order_id     = o.order_id
    WHERE full_table.status != 'cancelled'
    GROUP BY c.name, c.tier, c.country
) ranked
ORDER BY total_spend DESC
LIMIT 10
"""

# Step 1: Run EXPLAIN ANALYZE
# plan = con.execute(f"EXPLAIN ANALYZE {slow_top_customers}").fetchall()
# for row in plan: print(row[-1])

# Step 2: Write your optimised version here
# fast_top_customers = """
# SELECT ...
# """

# Step 3: Benchmark both and print the speedup
# Your solution here


---
## Day 8 key concepts recap

| Concept | What to remember |
|---|---|
| `EXPLAIN` | Shows the query plan without executing — free to run |
| `EXPLAIN ANALYZE` | Runs the query and annotates with actual rows + timing |
| Estimated vs actual rows | Large divergences = stale statistics → run `ANALYZE` |
| `ANALYZE tablename` | Refreshes per-column stats (min, max, distinct count, nulls) |
| `PRAGMA enable_profiling = 'json'` | Writes machine-readable profile JSON after each query |
| `SET threads = N` | Controls CPU parallelism for the current connection |
| Predicate pushdown | Filter early — push WHERE clauses into subquery/CTE scans |
| Column pruning | `SELECT a, b` not `SELECT *` — read only what you need |

> **Tip:** EXPLAIN ANALYZE is your best debugging tool — focus on operators where actual rows >> estimated rows. Large discrepancies often mean missing statistics; run ANALYZE on your table to refresh them.

---
## What's next
**Day 9** → Persistent databases, transactions, ACID guarantees, ATTACH, and safe upsert patterns.

Mark Day 8 complete in your [tracker](../index.html).